# Full-SPD Eight-Gaussians noise-parameter estimation


This notebook reproduces the full-SPD noise-parameter recovery and family-control results in the paper. Five independent noisy-data replicates each pair with one Hyvarinen score model. Three independently trained random-SPD Engressors supply learned posteriors. The four source combinations are crossed with four fitted noise families, producing the 16 reported family/source rows. The selected posterior and score checkpoints are included. `exp/train_parameter_estimation_models.sh` retrains them from their saved configurations. Generated data, fits, and tables are saved under `res/8gauss/parameter_estimation/random_spd_plugin/`.


## Imports and setup


In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)


In [ ]:
import math
import sys
from pathlib import Path

import pandas as pd
import torch
from omegaconf import OmegaConf
from torch.nn import functional as F


In [ ]:
CWD = Path.cwd().resolve()
ROOT_CANDIDATES = [
    CWD,
    *CWD.parents,
]
ROOT = next(
    (p for p in ROOT_CANDIDATES if (p / "utils").exists() and (p / "exp").exists()),
    None,
)
if ROOT is None:
    raise RuntimeError(f"Could not infer project root from {CWD}")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ENGRESSION_PATH = ROOT / "src" / "engression"
if str(ENGRESSION_PATH) not in sys.path:
    sys.path.insert(1, str(ENGRESSION_PATH))

print(ROOT)


In [ ]:
from utils.eight_gauss import eight_gaussians
from utils.experiment_io import format_mean_sd as mean_sd_string
from utils.experiment_io import (
    save_torch_atomic,
    set_reproducible_seed,
    write_csv_atomic,
    write_json_atomic,
)
from utils.noise_geometry import spd_2d_from_uv, uv_from_spd_2d
from utils.parameter_estimation import (
    evaluate_multiscale_noise_parameters,
    fit_multiscale_profiled_noise_parameters,
    profiled_score_reconstruction,
)
from utils.sampling import (
    SPDConditionedEngressorSampler,
    generalized_importance_references,
    sample_grouped_engressor_posterior,
    scaled_energy_tweedie_field_from_samples,
)
from utils.score_matching import (
    load_score_matching_checkpoint,
    predict_conditional_score,
    sample_noisy_eight_gaussians,
    score_diagnostics_by_sigma,
)


In [ ]:
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
MODEL_ROOT = ROOT / "models"
OUT_DIR = ROOT / "res" / "8gauss" / "parameter_estimation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(DEVICE))
    print(
        "GPU memory GB:",
        round(torch.cuda.get_device_properties(DEVICE).total_memory / 1024**3, 2),
    )


In [ ]:
IMPORTANCE_CHUNK = 32_768


## Oracle-score parameter estimation


**Experiment configuration**


In [ ]:
FULL_SPD_OUT_DIR = OUT_DIR / "random_spd_plugin"
FULL_SPD_FIT_DIR = FULL_SPD_OUT_DIR / "fits"
FULL_SPD_DATA_DIR = FULL_SPD_OUT_DIR / "paired_data"
FULL_SPD_OUT_DIR.mkdir(parents=True, exist_ok=True)
FULL_SPD_FIT_DIR.mkdir(parents=True, exist_ok=True)
FULL_SPD_DATA_DIR.mkdir(parents=True, exist_ok=True)

FULL_SPD_MODEL_SPECS = [
    {
        "model_name": "posterior_seed42",
        "training_seed": 42,
    },
    {
        "model_name": "posterior_seed43",
        "training_seed": 43,
    },
    {"model_name": "posterior_seed46", "training_seed": 46},
]
FULL_SPD_CHECKPOINT_NAME = "model.pt"
FULL_SPD_TRUE_BETA = 1.4
FULL_SPD_TRUE_LAMBDA = 1.8
FULL_SPD_TRUE_UV = torch.tensor([0.25, 0.35], device=DEVICE, dtype=DTYPE)
FULL_SPD_TRUE_SIGMA = spd_2d_from_uv(FULL_SPD_TRUE_UV)
FULL_SPD_SIGMAS = (0.25, 0.45, 0.75)
FULL_SPD_STD_CLEAN = 0.2
FULL_SPD_RADIUS = 2.0

FULL_SPD_N_DATA_REPLICATES = 5
FULL_SPD_N_PER_SIGMA = 384
FULL_SPD_N_PRIOR = 160_000
FULL_SPD_N_POSTERIOR = 256
FULL_SPD_N_STEPS = 2_200
FULL_SPD_BATCH_SIZE = 384
FULL_SPD_REFERENCE_QUERY_BATCH = 64
FULL_SPD_MODEL_QUERY_BATCH = 96
FULL_SPD_OPTIMIZER_LR = 1e-2
FULL_SPD_LOG_EVERY = 100
FULL_SPD_BASE_SEED = 202607220
FULL_SPD_ESTIMATION_SEED_OFFSET = 100
FULL_SPD_EVALUATION_SEED_OFFSET = 200
FULL_SPD_DATA_SEEDS = [
    FULL_SPD_BASE_SEED + 1_000 * index for index in range(FULL_SPD_N_DATA_REPLICATES)
]

FULL_SPD_RAW_PATH = FULL_SPD_OUT_DIR / "full_spd_fits_raw.csv"
FULL_SPD_HISTORY_PATH = FULL_SPD_OUT_DIR / "full_spd_histories.csv"
FULL_SPD_PER_MODEL_PATH = FULL_SPD_OUT_DIR / "full_spd_per_model_summary.csv"
FULL_SPD_SOURCE_SUMMARY_PATH = FULL_SPD_OUT_DIR / "full_spd_source_summary.csv"
FULL_SPD_CONFIG_PATH = FULL_SPD_OUT_DIR / "full_spd_config.json"
FULL_SPD_BUNDLE_PATH = FULL_SPD_OUT_DIR / "full_spd_results.pt"

full_spd_config = {
    "truth": {
        "beta": FULL_SPD_TRUE_BETA,
        "lambda": FULL_SPD_TRUE_LAMBDA,
        "u": 0.25,
        "v": 0.35,
    },
    "sigmas": list(FULL_SPD_SIGMAS),
    "std_clean": FULL_SPD_STD_CLEAN,
    "radius": FULL_SPD_RADIUS,
    "n_data_replicates": FULL_SPD_N_DATA_REPLICATES,
    "n_per_sigma": FULL_SPD_N_PER_SIGMA,
    "n_prior_per_independent_bank": FULL_SPD_N_PRIOR,
    "n_posterior_samples": FULL_SPD_N_POSTERIOR,
    "n_optimizer_steps": FULL_SPD_N_STEPS,
    "optimizer_lr": FULL_SPD_OPTIMIZER_LR,
    "data_seeds": FULL_SPD_DATA_SEEDS,
    "observation_seed_offsets": {
        "estimation": FULL_SPD_ESTIMATION_SEED_OFFSET,
        "evaluation": FULL_SPD_EVALUATION_SEED_OFFSET,
    },
    "model_specs": FULL_SPD_MODEL_SPECS,
    "score_bank_seed": FULL_SPD_BASE_SEED + 101,
    "posterior_bank_seed": FULL_SPD_BASE_SEED + 102,
    "heldout_score_bank_seed": FULL_SPD_BASE_SEED + 103,
    "heldout_posterior_bank_seed": FULL_SPD_BASE_SEED + 104,
}
write_json_atomic(full_spd_config, FULL_SPD_CONFIG_PATH)
full_spd_config


**Load the three posterior models**


In [ ]:
full_spd_model_cache = {}
for model_spec in FULL_SPD_MODEL_SPECS:
    model_name = model_spec["model_name"]
    model_dir = MODEL_ROOT / "parameter_estimation" / model_name
    checkpoint = model_dir / FULL_SPD_CHECKPOINT_NAME
    config = OmegaConf.load(model_dir / "config.yaml")

    if str(config.noise.condition) != "log_sigma_spd":
        raise ValueError(f"{model_name} is not conditioned on (log sigma, u, v)")
    if str(config.generalized.covariance.type) != "random_spd_2d":
        raise ValueError(f"{model_name} is not a random-SPD model")
    if float(config.generalized.covariance.radius_max) < float(
        torch.linalg.vector_norm(FULL_SPD_TRUE_UV)
    ):
        raise ValueError(f"{model_name} does not cover the requested full-SPD geometry")

    expected_config_values = {
        "std_clean": (float(config.data.std_clean), FULL_SPD_STD_CLEAN),
        "beta": (float(config.generalized.beta_noise), FULL_SPD_TRUE_BETA),
        "lambda": (float(config.generalized.lam), FULL_SPD_TRUE_LAMBDA),
        "training_seed": (
            int(config.training.seed),
            int(model_spec["training_seed"]),
        ),
    }
    for name, (observed, expected) in expected_config_values.items():
        if not math.isclose(
            float(observed),
            float(expected),
            rel_tol=0.0,
            abs_tol=1e-7,
        ):
            raise ValueError(
                f"{model_name}: {name}={observed} does not match {expected}"
            )

    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    engressor = torch.load(checkpoint, map_location=DEVICE, weights_only=False)
    if int(getattr(engressor, "condition_dim", -1)) != 3:
        raise ValueError(f"{model_name} does not have condition_dim=3")

    full_spd_model_cache[model_name] = {
        "sampler": SPDConditionedEngressorSampler(
            engressor,
            DEVICE,
            compile_model=False,
        ),
        "config": config,
        "model_dir": model_dir,
        "checkpoint": checkpoint,
    }

list(full_spd_model_cache)


**Generate paired observations, fit parameters, and evaluate held-out scores**


The estimation and held-out observations are independently sampled for each replicate. Oracle noisy scores and posterior samples use independent clean-particle banks. Learned posterior samples are saved with their fits so later comparisons use exactly the same draws.


In [ ]:
full_spd_prior_banks = {}
for bank_name, bank_seed in {
    "score": full_spd_config["score_bank_seed"],
    "posterior": full_spd_config["posterior_bank_seed"],
    "heldout_score": full_spd_config["heldout_score_bank_seed"],
    "heldout_posterior": full_spd_config["heldout_posterior_bank_seed"],
}.items():
    set_reproducible_seed(bank_seed)
    full_spd_prior_banks[bank_name] = eight_gaussians(
        FULL_SPD_N_PRIOR,
        radius=FULL_SPD_RADIUS,
        std=FULL_SPD_STD_CLEAN,
        device=DEVICE,
    ).to(dtype=DTYPE)

full_spd_rows = []
full_spd_history_frames = []

for replicate, data_seed in enumerate(FULL_SPD_DATA_SEEDS):
    estimation_observation_seed = int(data_seed + FULL_SPD_ESTIMATION_SEED_OFFSET)
    evaluation_observation_seed = int(data_seed + FULL_SPD_EVALUATION_SEED_OFFSET)
    data_path = FULL_SPD_DATA_DIR / f"paired_data_rep{replicate:02d}.pt"

    if data_path.exists():
        data_payload_cpu = torch.load(data_path, map_location="cpu", weights_only=False)

        expected_observation_seeds = {
            "estimation_observation_seed": estimation_observation_seed,
            "evaluation_observation_seed": evaluation_observation_seed,
        }
        for seed_name, expected_seed in expected_observation_seeds.items():
            if int(data_payload_cpu.get(seed_name, -1)) != expected_seed:
                raise RuntimeError(f"Unexpected {seed_name} in {data_path}")

        data_payload = {
            key: (
                value.to(device=DEVICE, dtype=DTYPE)
                if torch.is_tensor(value)
                else value
            )
            for key, value in data_payload_cpu.items()
        }
    else:
        set_reproducible_seed(estimation_observation_seed)
        y_estimation, sigma_estimation, _ = sample_noisy_eight_gaussians(
            FULL_SPD_N_PER_SIGMA * len(FULL_SPD_SIGMAS),
            sigmas=FULL_SPD_SIGMAS,
            beta=FULL_SPD_TRUE_BETA,
            lam=FULL_SPD_TRUE_LAMBDA,
            Sigma=FULL_SPD_TRUE_SIGMA,
            radius=FULL_SPD_RADIUS,
            std=FULL_SPD_STD_CLEAN,
            balanced=True,
            shuffle=False,
            device=DEVICE,
            dtype=DTYPE,
        )
        set_reproducible_seed(evaluation_observation_seed)
        y_evaluation, sigma_evaluation, _ = sample_noisy_eight_gaussians(
            FULL_SPD_N_PER_SIGMA * len(FULL_SPD_SIGMAS),
            sigmas=FULL_SPD_SIGMAS,
            beta=FULL_SPD_TRUE_BETA,
            lam=FULL_SPD_TRUE_LAMBDA,
            Sigma=FULL_SPD_TRUE_SIGMA,
            radius=FULL_SPD_RADIUS,
            std=FULL_SPD_STD_CLEAN,
            balanced=True,
            shuffle=False,
            device=DEVICE,
            dtype=DTYPE,
        )

        set_reproducible_seed(data_seed + 300)
        score_estimation, oracle_posterior_estimation = (
            generalized_importance_references(
                y_estimation,
                sigma_estimation,
                full_spd_prior_banks["score"],
                posterior_prior=full_spd_prior_banks["posterior"],
                beta=FULL_SPD_TRUE_BETA,
                lam=FULL_SPD_TRUE_LAMBDA,
                Sigma=FULL_SPD_TRUE_SIGMA,
                n_samples=FULL_SPD_N_POSTERIOR,
                chunk_size=IMPORTANCE_CHUNK,
                query_batch_size=FULL_SPD_REFERENCE_QUERY_BATCH,
            )
        )
        set_reproducible_seed(data_seed + 301)
        score_evaluation, oracle_posterior_evaluation = (
            generalized_importance_references(
                y_evaluation,
                sigma_evaluation,
                full_spd_prior_banks["heldout_score"],
                posterior_prior=full_spd_prior_banks["heldout_posterior"],
                beta=FULL_SPD_TRUE_BETA,
                lam=FULL_SPD_TRUE_LAMBDA,
                Sigma=FULL_SPD_TRUE_SIGMA,
                n_samples=FULL_SPD_N_POSTERIOR,
                chunk_size=IMPORTANCE_CHUNK,
                query_batch_size=FULL_SPD_REFERENCE_QUERY_BATCH,
            )
        )

        data_payload = {
            "y_estimation": y_estimation,
            "sigma_estimation": sigma_estimation,
            "score_estimation": score_estimation,
            "oracle_posterior_estimation": oracle_posterior_estimation,
            "y_evaluation": y_evaluation,
            "sigma_evaluation": sigma_evaluation,
            "score_evaluation": score_evaluation,
            "oracle_posterior_evaluation": oracle_posterior_evaluation,
            "data_seed": int(data_seed),
            "estimation_observation_seed": estimation_observation_seed,
            "evaluation_observation_seed": evaluation_observation_seed,
        }
        save_torch_atomic(
            {
                key: value.detach().cpu() if torch.is_tensor(value) else value
                for key, value in data_payload.items()
            },
            data_path,
        )

    estimation_data = {
        "y": data_payload["y_estimation"],
        "sigma": data_payload["sigma_estimation"],
        "score": data_payload["score_estimation"],
    }
    evaluation_data = {
        "y": data_payload["y_evaluation"],
        "sigma": data_payload["sigma_evaluation"],
        "score": data_payload["score_evaluation"],
    }
    optimizer_seed = int(data_seed) + 400

    source_specs = [
        {
            "posterior_source": "importance_posterior",
            "model_name": "independent_importance_reference",
            "training_seed": -1,
            "checkpoint": "independent_importance_bank",
            "posterior_seed": int(data_seed) + 300,
            "fit_path": FULL_SPD_FIT_DIR
            / f"importance_posterior_rep{replicate:02d}.pt",
            "posterior_estimation": data_payload["oracle_posterior_estimation"],
            "posterior_evaluation": data_payload["oracle_posterior_evaluation"],
        }
    ]
    for model_spec in FULL_SPD_MODEL_SPECS:
        training_seed = int(model_spec["training_seed"])
        source_specs.append(
            {
                "posterior_source": "learned_posterior",
                "model_name": model_spec["model_name"],
                "training_seed": training_seed,
                "checkpoint": full_spd_model_cache[model_spec["model_name"]][
                    "checkpoint"
                ],
                "posterior_seed": int(data_seed) + 10_000 + training_seed,
                "fit_path": FULL_SPD_FIT_DIR
                / f"learned_seed{training_seed}_rep{replicate:02d}.pt",
            }
        )

    for source_spec in source_specs:
        fit_path = source_spec["fit_path"]
        if fit_path.exists():
            fit_artifact = torch.load(fit_path, map_location="cpu", weights_only=False)
            row = fit_artifact["row"]
            history = pd.DataFrame(fit_artifact["history"])
        else:
            posterior_seed = int(source_spec["posterior_seed"])
            if source_spec["posterior_source"] == "learned_posterior":
                sampler = full_spd_model_cache[source_spec["model_name"]]["sampler"]
                set_reproducible_seed(posterior_seed)
                posterior_estimation = sample_grouped_engressor_posterior(
                    sampler,
                    estimation_data["y"],
                    estimation_data["sigma"],
                    FULL_SPD_N_POSTERIOR,
                    batch_size=FULL_SPD_MODEL_QUERY_BATCH,
                    sample_layout="kbd",
                    condition_args=(FULL_SPD_TRUE_UV,),
                )
                set_reproducible_seed(posterior_seed + 1)
                posterior_evaluation = sample_grouped_engressor_posterior(
                    sampler,
                    evaluation_data["y"],
                    evaluation_data["sigma"],
                    FULL_SPD_N_POSTERIOR,
                    batch_size=FULL_SPD_MODEL_QUERY_BATCH,
                    sample_layout="kbd",
                    condition_args=(FULL_SPD_TRUE_UV,),
                )
            else:
                posterior_estimation = source_spec["posterior_estimation"]
                posterior_evaluation = source_spec["posterior_evaluation"]

            fit = fit_multiscale_profiled_noise_parameters(
                estimation_data["y"],
                estimation_data["sigma"],
                posterior_estimation,
                score_estimator=estimation_data["score"],
                covariance_type="full",
                beta_min=0.5,
                beta_init=1.6,
                init_sigma=torch.eye(2, device=DEVICE, dtype=DTYPE),
                lambda_bounds=(1e-4, 10.0),
                n_steps=FULL_SPD_N_STEPS,
                batch_size=FULL_SPD_BATCH_SIZE,
                lr=FULL_SPD_OPTIMIZER_LR,
                grad_clip=10.0,
                log_every=FULL_SPD_LOG_EVERY,
                seed=optimizer_seed,
                device=DEVICE,
                sample_layout="bkd",
            )
            evaluation = evaluate_multiscale_noise_parameters(
                evaluation_data["y"],
                evaluation_data["sigma"],
                posterior_evaluation,
                score_estimator=evaluation_data["score"],
                reference_score=evaluation_data["score"],
                beta=fit.beta,
                lam=fit.lam,
                Sigma=fit.Sigma,
                batch_size=FULL_SPD_BATCH_SIZE,
                sample_layout="bkd",
            )

            fitted_uv = uv_from_spd_2d(fit.Sigma)
            score_power = (
                evaluation_data["score"].square().sum(dim=1).mean().clamp_min(1e-12)
            )
            fitted_eigenvalues = torch.linalg.eigvalsh(fit.Sigma)
            row = {
                "posterior_source": source_spec["posterior_source"],
                "model_name": source_spec["model_name"],
                "training_seed": int(source_spec["training_seed"]),
                "replicate": int(replicate),
                "data_seed": int(data_seed),
                "estimation_observation_seed": estimation_observation_seed,
                "evaluation_observation_seed": evaluation_observation_seed,
                "posterior_seed": posterior_seed,
                "optimizer_seed": optimizer_seed,
                "checkpoint": str(source_spec["checkpoint"]),
                "beta": float(fit.beta),
                "lambda": float(fit.lam),
                "u": float(fitted_uv[0]),
                "v": float(fitted_uv[1]),
                "sigma_00": float(fit.Sigma[0, 0]),
                "sigma_01": float(fit.Sigma[0, 1]),
                "sigma_11": float(fit.Sigma[1, 1]),
                "abs_beta_error": abs(float(fit.beta) - FULL_SPD_TRUE_BETA),
                "abs_lambda_error": abs(float(fit.lam) - FULL_SPD_TRUE_LAMBDA),
                "abs_u_error": abs(float(fitted_uv[0]) - float(FULL_SPD_TRUE_UV[0])),
                "abs_v_error": abs(float(fitted_uv[1]) - float(FULL_SPD_TRUE_UV[1])),
                "sigma_frobenius_error": float(
                    torch.linalg.matrix_norm(fit.Sigma - FULL_SPD_TRUE_SIGMA.cpu())
                ),
                "fitted_condition_number": float(
                    fitted_eigenvalues.max() / fitted_eigenvalues.min()
                ),
                "estimation_objective": float(fit.objective),
                "heldout_identity_residual": float(evaluation["identity_residual"]),
                "heldout_score_mse": float(evaluation["reference_mse"]),
                "heldout_score_nmse": float(evaluation["reference_mse"] / score_power),
                "heldout_score_cosine": float(
                    F.cosine_similarity(
                        evaluation["prediction"],
                        evaluation_data["score"],
                        dim=1,
                        eps=1e-8,
                    ).mean()
                ),
            }
            history = pd.DataFrame(fit.history)
            history["posterior_source"] = source_spec["posterior_source"]
            history["model_name"] = source_spec["model_name"]
            history["training_seed"] = int(source_spec["training_seed"])
            history["replicate"] = int(replicate)

            fit_payload = {
                "row": row,
                "history": history.to_dict("records"),
                "fit": fit.as_dict(),
            }
            if source_spec["posterior_source"] == "learned_posterior":
                fit_payload.update(
                    {
                        "posterior_estimation": (posterior_estimation.detach().cpu()),
                        "posterior_evaluation": (posterior_evaluation.detach().cpu()),
                    }
                )
            save_torch_atomic(fit_payload, fit_path)

            if source_spec["posterior_source"] == "learned_posterior":
                del posterior_estimation, posterior_evaluation

        full_spd_rows.append(row)
        full_spd_history_frames.append(history)
        print(
            f"replicate={replicate} source={source_spec['model_name']} "
            f"beta={row['beta']:.4f} lambda={row['lambda']:.4f} "
            f"u={row['u']:.4f} v={row['v']:.4f}"
        )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

full_spd_raw = (
    pd.DataFrame(full_spd_rows)
    .sort_values(["replicate", "posterior_source", "training_seed"])
    .reset_index(drop=True)
)
full_spd_histories = pd.concat(full_spd_history_frames, ignore_index=True)

full_spd_metrics = [
    "beta",
    "lambda",
    "u",
    "v",
    "abs_beta_error",
    "abs_lambda_error",
    "abs_u_error",
    "abs_v_error",
    "sigma_frobenius_error",
    "estimation_objective",
    "heldout_identity_residual",
    "heldout_score_mse",
    "heldout_score_nmse",
    "heldout_score_cosine",
]
full_spd_per_model_rows = []
full_spd_group_columns = [
    "posterior_source",
    "model_name",
    "training_seed",
]
for group_values, group in full_spd_raw.groupby(
    full_spd_group_columns,
    sort=False,
):
    summary_row = dict(zip(full_spd_group_columns, group_values))
    summary_row["n_data_replicates"] = int(group["replicate"].nunique())
    for metric in full_spd_metrics:
        values = group[metric].to_numpy(dtype=float)
        summary_row[f"{metric}_mean"] = float(values.mean())
        summary_row[f"{metric}_sd"] = (
            float(values.std(ddof=1)) if len(values) > 1 else 0.0
        )
    full_spd_per_model_rows.append(summary_row)
full_spd_per_model = pd.DataFrame(full_spd_per_model_rows)

full_spd_source_rows = []
for posterior_source in ("importance_posterior", "learned_posterior"):
    if posterior_source == "importance_posterior":
        units = full_spd_raw[full_spd_raw["posterior_source"] == posterior_source]
        uncertainty_unit = "paired data and posterior Monte Carlo replicate"
        metric_columns = {metric: metric for metric in full_spd_metrics}
    else:
        units = full_spd_raw[full_spd_raw["posterior_source"] == posterior_source]
        uncertainty_unit = (
            "fit crossing a posterior-model training seed with a paired data replicate"
        )
        metric_columns = {metric: metric for metric in full_spd_metrics}

    summary_row = {
        "posterior_source": posterior_source,
        "uncertainty_unit": uncertainty_unit,
        "n_units": int(len(units)),
    }
    for metric, column in metric_columns.items():
        values = units[column].to_numpy(dtype=float)
        summary_row[f"{metric}_mean"] = float(values.mean())
        summary_row[f"{metric}_sd"] = (
            float(values.std(ddof=1)) if len(values) > 1 else 0.0
        )
        summary_row[f"{metric}_sem"] = summary_row[f"{metric}_sd"] / math.sqrt(
            len(values)
        )
    full_spd_source_rows.append(summary_row)
full_spd_source_summary = pd.DataFrame(full_spd_source_rows)

write_csv_atomic(full_spd_raw, FULL_SPD_RAW_PATH)
write_csv_atomic(full_spd_histories, FULL_SPD_HISTORY_PATH)
write_csv_atomic(full_spd_per_model, FULL_SPD_PER_MODEL_PATH)
write_csv_atomic(full_spd_source_summary, FULL_SPD_SOURCE_SUMMARY_PATH)
save_torch_atomic(
    {
        "config": full_spd_config,
        "raw": full_spd_raw.to_dict("records"),
        "per_model": full_spd_per_model.to_dict("records"),
        "source_summary": full_spd_source_summary.to_dict("records"),
    },
    FULL_SPD_BUNDLE_PATH,
)
full_spd_source_summary


**Load the oracle-score results**


In [ ]:
required_full_spd_paths = [
    FULL_SPD_CONFIG_PATH,
    FULL_SPD_RAW_PATH,
    FULL_SPD_HISTORY_PATH,
    FULL_SPD_PER_MODEL_PATH,
    FULL_SPD_SOURCE_SUMMARY_PATH,
    FULL_SPD_BUNDLE_PATH,
]
missing_full_spd_paths = [
    str(path) for path in required_full_spd_paths if not path.exists()
]
if missing_full_spd_paths:
    raise FileNotFoundError(
        f"Run the full-SPD recovery cell first; missing {missing_full_spd_paths}"
    )

full_spd_raw = pd.read_csv(FULL_SPD_RAW_PATH)
full_spd_histories = pd.read_csv(FULL_SPD_HISTORY_PATH)
full_spd_per_model = pd.read_csv(FULL_SPD_PER_MODEL_PATH)
full_spd_source_summary = pd.read_csv(FULL_SPD_SOURCE_SUMMARY_PATH)


## Learned-score parameter estimation


**Score model configuration**


In [ ]:
FULL_SPD_HYVARINEN_CAMPAIGN = "full_spd_parameter_estimation"
FULL_SPD_HYVARINEN_TRAIN_DIR = MODEL_ROOT / "parameter_estimation"
FULL_SPD_HYVARINEN_FIT_DIR = FULL_SPD_OUT_DIR / "hyvarinen_fits"
FULL_SPD_HYVARINEN_FIT_DIR.mkdir(parents=True, exist_ok=True)

FULL_SPD_HYVARINEN_SCORE_SPECS = [
    {
        "training_seed": score_seed,
        "run_name": f"score_seed{score_seed}",
    }
    for score_seed in (10042, 11042, 12042, 13042, 14042)
]

FULL_SPD_HYVARINEN_RAW_PATH = FULL_SPD_OUT_DIR / "full_spd_hyvarinen_fits_raw.csv"
FULL_SPD_HYVARINEN_HISTORY_PATH = FULL_SPD_OUT_DIR / "full_spd_hyvarinen_histories.csv"
FULL_SPD_HYVARINEN_DIAGNOSTIC_PATH = (
    FULL_SPD_OUT_DIR / "full_spd_hyvarinen_score_diagnostics.csv"
)
FULL_SPD_LADDER_RAW_PATH = (
    FULL_SPD_OUT_DIR / "full_spd_parameter_estimation_ladder_raw.csv"
)
FULL_SPD_LADDER_PER_MODEL_PATH = (
    FULL_SPD_OUT_DIR / "full_spd_parameter_estimation_ladder_per_model.csv"
)
FULL_SPD_LADDER_SUMMARY_PATH = (
    FULL_SPD_OUT_DIR / "full_spd_parameter_estimation_ladder_summary.csv"
)
FULL_SPD_LADDER_BUNDLE_PATH = (
    FULL_SPD_OUT_DIR / "full_spd_parameter_estimation_ladder.pt"
)

full_spd_hyvarinen_config = {
    "campaign": FULL_SPD_HYVARINEN_CAMPAIGN,
    "score_models": FULL_SPD_HYVARINEN_SCORE_SPECS,
    "paired_data_seeds": FULL_SPD_DATA_SEEDS,
    "posterior_models": FULL_SPD_MODEL_SPECS,
    "truth": {
        "beta": FULL_SPD_TRUE_BETA,
        "lambda": FULL_SPD_TRUE_LAMBDA,
        "u": float(FULL_SPD_TRUE_UV[0]),
        "v": float(FULL_SPD_TRUE_UV[1]),
    },
    "sigmas": list(FULL_SPD_SIGMAS),
    "reuses_oracle_stage_posterior_samples": True,
}
full_spd_hyvarinen_config


**Load the five matched Hyvarinen estimators**


In [ ]:
if len(FULL_SPD_HYVARINEN_SCORE_SPECS) != len(FULL_SPD_DATA_SEEDS):
    raise ValueError(
        "The score-model seeds must pair one-to-one with the five data replicates"
    )

full_spd_hyvarinen_model_cache = {}
full_spd_hyvarinen_model_rows = []
for score_spec in FULL_SPD_HYVARINEN_SCORE_SPECS:
    score_seed = int(score_spec["training_seed"])
    score_dir = FULL_SPD_HYVARINEN_TRAIN_DIR / score_spec["run_name"]
    score_checkpoint = score_dir / "model.pt"
    score_config_path = score_dir / "config.yaml"
    if not score_checkpoint.is_file():
        raise FileNotFoundError(score_checkpoint)
    if not score_config_path.is_file():
        raise FileNotFoundError(score_config_path)

    score_config = OmegaConf.load(score_config_path)
    expected_scalar_values = {
        "data.std_clean": (float(score_config.data.std_clean), FULL_SPD_STD_CLEAN),
        "generalized.beta_noise": (
            float(score_config.generalized.beta_noise),
            FULL_SPD_TRUE_BETA,
        ),
        "generalized.lam": (float(score_config.generalized.lam), FULL_SPD_TRUE_LAMBDA),
        "training.seed": (int(score_config.training.seed), score_seed),
    }
    for field, (observed, expected) in expected_scalar_values.items():
        if not math.isclose(
            float(observed), float(expected), rel_tol=0.0, abs_tol=1e-7
        ):
            raise ValueError(
                f"{score_spec['run_name']}: {field}={observed}, expected {expected}"
            )
    if str(score_config.training.objective) != "exact_hyvarinen":
        raise ValueError(
            f"{score_spec['run_name']} was not trained with exact Hyvarinen score matching"
        )
    if str(score_config.generalized.covariance.type) != "fixed_spd_2d":
        raise ValueError(f"{score_spec['run_name']} does not use fixed full-SPD noise")
    configured_sigmas = tuple(float(value) for value in score_config.noise.sigmas)
    if configured_sigmas != tuple(float(value) for value in FULL_SPD_SIGMAS):
        raise ValueError(f"{score_spec['run_name']} has sigma grid {configured_sigmas}")
    configured_uv = torch.as_tensor(
        list(score_config.generalized.covariance.uv), device=DEVICE, dtype=DTYPE
    )
    if not torch.allclose(configured_uv, FULL_SPD_TRUE_UV, rtol=0.0, atol=1e-7):
        raise ValueError(
            f"{score_spec['run_name']} has covariance coordinates {configured_uv.tolist()}"
        )

    score_model, score_payload = load_score_matching_checkpoint(
        score_checkpoint, map_location=DEVICE
    )
    score_model = score_model.to(device=DEVICE, dtype=DTYPE)
    score_model.eval()
    full_spd_hyvarinen_model_cache[score_seed] = {
        "model": score_model,
        "checkpoint": score_checkpoint,
        "config_path": score_config_path,
        "payload": score_payload,
    }
    full_spd_hyvarinen_model_rows.append(
        {
            "score_training_seed": score_seed,
            "checkpoint": str(score_checkpoint),
        }
    )

pd.DataFrame(full_spd_hyvarinen_model_rows)


**Fit four source combinations on paired data**


The two score sources are the oracle importance-sampled score and a separately trained Hyvarinen score estimator. The two posterior sources are importance sampling and a learned Engressor. Each source combination uses the same estimation and held-out observations within a replicate.


In [ ]:
if not FULL_SPD_RAW_PATH.exists():
    raise FileNotFoundError(
        "Run the oracle/learned-posterior stage first; this section reuses its oracle rows and posterior samples"
    )

full_spd_hyvarinen_rows = []
full_spd_hyvarinen_history_frames = []
full_spd_hyvarinen_diagnostic_rows = []

for replicate, (data_seed, score_spec) in enumerate(
    zip(FULL_SPD_DATA_SEEDS, FULL_SPD_HYVARINEN_SCORE_SPECS)
):
    score_seed = int(score_spec["training_seed"])
    score_model_record = full_spd_hyvarinen_model_cache[score_seed]
    data_path = FULL_SPD_DATA_DIR / f"paired_data_rep{replicate:02d}.pt"
    if not data_path.exists():
        raise FileNotFoundError(data_path)
    data_payload_cpu = torch.load(data_path, map_location="cpu", weights_only=False)
    data_payload = {
        key: value.to(device=DEVICE, dtype=DTYPE) if torch.is_tensor(value) else value
        for key, value in data_payload_cpu.items()
    }

    hyvarinen_score_estimation = predict_conditional_score(
        score_model_record["model"],
        data_payload["y_estimation"],
        data_payload["sigma_estimation"],
        batch_size=8192,
    ).detach()
    hyvarinen_score_evaluation = predict_conditional_score(
        score_model_record["model"],
        data_payload["y_evaluation"],
        data_payload["sigma_evaluation"],
        batch_size=8192,
    ).detach()

    for diagnostic in score_diagnostics_by_sigma(
        hyvarinen_score_evaluation,
        data_payload["score_evaluation"],
        data_payload["sigma_evaluation"],
        sigmas=FULL_SPD_SIGMAS,
    ):
        full_spd_hyvarinen_diagnostic_rows.append(
            {
                "replicate": int(replicate),
                "data_seed": int(data_seed),
                "score_training_seed": score_seed,
                "score_checkpoint": str(score_model_record["checkpoint"]),
                **diagnostic,
            }
        )

    posterior_specs = [
        {
            "posterior_source": "importance_posterior",
            "model_name": "independent_importance_reference",
            "training_seed": -1,
            "checkpoint": "independent_importance_bank",
            "posterior_seed": int(data_seed) + 300,
            "posterior_estimation": data_payload["oracle_posterior_estimation"],
            "posterior_evaluation": data_payload["oracle_posterior_evaluation"],
        }
    ]
    for model_spec in FULL_SPD_MODEL_SPECS:
        posterior_training_seed = int(model_spec["training_seed"])
        oracle_stage_fit_path = (
            FULL_SPD_FIT_DIR
            / f"learned_seed{posterior_training_seed}_rep{replicate:02d}.pt"
        )
        if not oracle_stage_fit_path.exists():
            raise FileNotFoundError(
                f"Run the the oracle/learned-posterior stage learned-posterior fit first; missing {oracle_stage_fit_path}"
            )
        oracle_stage_fit = torch.load(
            oracle_stage_fit_path, map_location="cpu", weights_only=False
        )
        if (
            "posterior_estimation" not in oracle_stage_fit
            or "posterior_evaluation" not in oracle_stage_fit
        ):
            raise RuntimeError(
                f"the oracle/learned-posterior stage fit does not contain saved posterior samples: {oracle_stage_fit_path}"
            )
        posterior_specs.append(
            {
                "posterior_source": "learned_posterior",
                "model_name": model_spec["model_name"],
                "training_seed": posterior_training_seed,
                "checkpoint": (
                    ROOT
                    / "models"
                    / "parameter_estimation"
                    / model_spec["model_name"]
                    / FULL_SPD_CHECKPOINT_NAME
                ),
                "posterior_seed": int(data_seed) + 10_000 + posterior_training_seed,
                "posterior_estimation": oracle_stage_fit["posterior_estimation"].to(
                    device=DEVICE, dtype=DTYPE
                ),
                "posterior_evaluation": oracle_stage_fit["posterior_evaluation"].to(
                    device=DEVICE, dtype=DTYPE
                ),
            }
        )

    oracle_score_power = (
        data_payload["score_evaluation"].square().sum(dim=1).mean().clamp_min(1e-12)
    )
    score_estimator_mse = (
        (hyvarinen_score_evaluation - data_payload["score_evaluation"])
        .square()
        .sum(dim=1)
        .mean()
    )
    score_estimator_cosine = F.cosine_similarity(
        hyvarinen_score_evaluation,
        data_payload["score_evaluation"],
        dim=1,
        eps=1e-8,
    ).mean()
    optimizer_seed = int(data_seed) + 400

    for posterior_spec in posterior_specs:
        posterior_training_seed = int(posterior_spec["training_seed"])
        posterior_tag = (
            "importance"
            if posterior_spec["posterior_source"] == "importance_posterior"
            else f"learned_seed{posterior_training_seed}"
        )
        fit_path = FULL_SPD_HYVARINEN_FIT_DIR / (
            f"hyvarinen_score_seed{score_seed}_{posterior_tag}_rep{replicate:02d}.pt"
        )
        if fit_path.exists():
            fit_artifact = torch.load(fit_path, map_location="cpu", weights_only=False)
            row = fit_artifact["row"]
            history = pd.DataFrame(fit_artifact["history"])
        else:
            fit = fit_multiscale_profiled_noise_parameters(
                data_payload["y_estimation"],
                data_payload["sigma_estimation"],
                posterior_spec["posterior_estimation"],
                score_estimator=hyvarinen_score_estimation,
                covariance_type="full",
                beta_min=0.5,
                beta_init=1.6,
                init_sigma=torch.eye(2, device=DEVICE, dtype=DTYPE),
                lambda_bounds=(1e-4, 10.0),
                n_steps=FULL_SPD_N_STEPS,
                batch_size=FULL_SPD_BATCH_SIZE,
                lr=FULL_SPD_OPTIMIZER_LR,
                grad_clip=10.0,
                log_every=FULL_SPD_LOG_EVERY,
                seed=optimizer_seed,
                device=DEVICE,
                sample_layout="bkd",
            )
            evaluation = evaluate_multiscale_noise_parameters(
                data_payload["y_evaluation"],
                data_payload["sigma_evaluation"],
                posterior_spec["posterior_evaluation"],
                score_estimator=hyvarinen_score_evaluation,
                reference_score=data_payload["score_evaluation"],
                beta=fit.beta,
                lam=fit.lam,
                Sigma=fit.Sigma,
                batch_size=FULL_SPD_BATCH_SIZE,
                sample_layout="bkd",
            )
            fitted_uv = uv_from_spd_2d(fit.Sigma)
            fitted_eigenvalues = torch.linalg.eigvalsh(fit.Sigma)
            row = {
                "score_source": "hyvarinen_score",
                "posterior_source": posterior_spec["posterior_source"],
                "model_name": posterior_spec["model_name"],
                "training_seed": posterior_training_seed,
                "score_training_seed": score_seed,
                "replicate": int(replicate),
                "data_seed": int(data_seed),
                "estimation_observation_seed": int(
                    data_payload["estimation_observation_seed"]
                ),
                "evaluation_observation_seed": int(
                    data_payload["evaluation_observation_seed"]
                ),
                "posterior_seed": int(posterior_spec["posterior_seed"]),
                "optimizer_seed": optimizer_seed,
                "checkpoint": str(posterior_spec["checkpoint"]),
                "score_checkpoint": str(score_model_record["checkpoint"]),
                "beta": float(fit.beta),
                "lambda": float(fit.lam),
                "u": float(fitted_uv[0]),
                "v": float(fitted_uv[1]),
                "sigma_00": float(fit.Sigma[0, 0]),
                "sigma_01": float(fit.Sigma[0, 1]),
                "sigma_11": float(fit.Sigma[1, 1]),
                "abs_beta_error": abs(float(fit.beta) - FULL_SPD_TRUE_BETA),
                "abs_lambda_error": abs(float(fit.lam) - FULL_SPD_TRUE_LAMBDA),
                "abs_u_error": abs(float(fitted_uv[0]) - float(FULL_SPD_TRUE_UV[0])),
                "abs_v_error": abs(float(fitted_uv[1]) - float(FULL_SPD_TRUE_UV[1])),
                "sigma_frobenius_error": float(
                    torch.linalg.matrix_norm(fit.Sigma - FULL_SPD_TRUE_SIGMA.cpu())
                ),
                "fitted_condition_number": float(
                    fitted_eigenvalues.max() / fitted_eigenvalues.min()
                ),
                "estimation_objective": float(fit.objective),
                "heldout_identity_residual": float(evaluation["identity_residual"]),
                "heldout_score_mse": float(evaluation["reference_mse"]),
                "heldout_score_nmse": float(
                    evaluation["reference_mse"] / oracle_score_power
                ),
                "heldout_score_cosine": float(
                    F.cosine_similarity(
                        evaluation["prediction"],
                        data_payload["score_evaluation"],
                        dim=1,
                        eps=1e-8,
                    ).mean()
                ),
                "score_estimator_mse": float(score_estimator_mse),
                "score_estimator_nmse": float(score_estimator_mse / oracle_score_power),
                "score_estimator_cosine": float(score_estimator_cosine),
            }
            history = pd.DataFrame(fit.history)
            history["score_source"] = "hyvarinen_score"
            history["posterior_source"] = posterior_spec["posterior_source"]
            history["model_name"] = posterior_spec["model_name"]
            history["training_seed"] = posterior_training_seed
            history["score_training_seed"] = score_seed
            history["replicate"] = int(replicate)
            save_torch_atomic(
                {
                    "row": row,
                    "history": history.to_dict("records"),
                    "fit": fit.as_dict(),
                },
                fit_path,
            )

        full_spd_hyvarinen_rows.append(row)
        full_spd_hyvarinen_history_frames.append(history)
        print(
            f"replicate={replicate} score_seed={score_seed} "
            f"posterior={posterior_spec['model_name']} beta={row['beta']:.4f} "
            f"lambda={row['lambda']:.4f} u={row['u']:.4f} v={row['v']:.4f}"
        )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

full_spd_hyvarinen_raw = (
    pd.DataFrame(full_spd_hyvarinen_rows)
    .sort_values(["replicate", "posterior_source", "training_seed"])
    .reset_index(drop=True)
)
full_spd_hyvarinen_histories = pd.concat(
    full_spd_hyvarinen_history_frames, ignore_index=True
)
full_spd_hyvarinen_diagnostics = (
    pd.DataFrame(full_spd_hyvarinen_diagnostic_rows)
    .sort_values(["replicate", "sigma"])
    .reset_index(drop=True)
)

full_spd_oracle_raw = pd.read_csv(FULL_SPD_RAW_PATH)
full_spd_oracle_raw["score_source"] = "oracle_score"
full_spd_oracle_raw["score_training_seed"] = -1
full_spd_oracle_raw["score_checkpoint"] = "independent_importance_score_reference"
full_spd_oracle_raw["score_estimator_mse"] = 0.0
full_spd_oracle_raw["score_estimator_nmse"] = 0.0
full_spd_oracle_raw["score_estimator_cosine"] = 1.0
full_spd_ladder_raw = pd.concat(
    [full_spd_oracle_raw, full_spd_hyvarinen_raw], ignore_index=True, sort=False
)

full_spd_ladder_metrics = [
    "beta",
    "lambda",
    "u",
    "v",
    "abs_beta_error",
    "abs_lambda_error",
    "abs_u_error",
    "abs_v_error",
    "sigma_frobenius_error",
    "estimation_objective",
    "heldout_identity_residual",
    "heldout_score_mse",
    "heldout_score_nmse",
    "heldout_score_cosine",
    "score_estimator_mse",
    "score_estimator_nmse",
    "score_estimator_cosine",
]
full_spd_ladder_per_model_rows = []
full_spd_ladder_group_columns = [
    "score_source",
    "posterior_source",
    "model_name",
    "training_seed",
]
for group_values, group in full_spd_ladder_raw.groupby(
    full_spd_ladder_group_columns, sort=False
):
    summary_row = dict(zip(full_spd_ladder_group_columns, group_values))
    summary_row["n_paired_replicates"] = int(group["replicate"].nunique())
    for metric in full_spd_ladder_metrics:
        values = group[metric].to_numpy(dtype=float)
        summary_row[f"{metric}_mean"] = float(values.mean())
        summary_row[f"{metric}_sd"] = (
            float(values.std(ddof=1)) if len(values) > 1 else 0.0
        )
    full_spd_ladder_per_model_rows.append(summary_row)
full_spd_ladder_per_model = pd.DataFrame(full_spd_ladder_per_model_rows)

full_spd_ladder_source_order = [
    ("oracle_score", "importance_posterior", "Oracle score / importance posterior"),
    (
        "hyvarinen_score",
        "importance_posterior",
        "Hyvarinen score / importance posterior",
    ),
    ("oracle_score", "learned_posterior", "Oracle score / learned posterior"),
    ("hyvarinen_score", "learned_posterior", "Hyvarinen score / learned posterior"),
]
full_spd_ladder_summary_rows = []
for score_source, posterior_source, source_label in full_spd_ladder_source_order:
    if posterior_source == "importance_posterior":
        units = full_spd_ladder_raw[
            (full_spd_ladder_raw["score_source"] == score_source)
            & (full_spd_ladder_raw["posterior_source"] == posterior_source)
        ]
        metric_columns = {metric: metric for metric in full_spd_ladder_metrics}
        summary_unit = (
            "five separately generated data sets"
            if score_source == "oracle_score"
            else "five independently trained score models, each paired with a separately generated data set"
        )
    else:
        units = full_spd_ladder_raw[
            (full_spd_ladder_raw["score_source"] == score_source)
            & (full_spd_ladder_raw["posterior_source"] == posterior_source)
        ]
        metric_columns = {metric: metric for metric in full_spd_ladder_metrics}
        summary_unit = (
            "fifteen fits crossing three independently trained posterior models"
            " with the five paired data runs"
        )

    summary_row = {
        "score_source": score_source,
        "posterior_source": posterior_source,
        "source_label": source_label,
        "summary_unit": summary_unit,
        "n_units": int(len(units)),
    }
    for metric, column in metric_columns.items():
        values = units[column].to_numpy(dtype=float)
        summary_row[f"{metric}_mean"] = float(values.mean())
        summary_row[f"{metric}_sd"] = (
            float(values.std(ddof=1)) if len(values) > 1 else 0.0
        )
        summary_row[f"{metric}_sem"] = summary_row[f"{metric}_sd"] / math.sqrt(
            len(values)
        )
    full_spd_ladder_summary_rows.append(summary_row)
full_spd_ladder_summary = pd.DataFrame(full_spd_ladder_summary_rows)

write_csv_atomic(full_spd_hyvarinen_raw, FULL_SPD_HYVARINEN_RAW_PATH)
write_csv_atomic(full_spd_hyvarinen_histories, FULL_SPD_HYVARINEN_HISTORY_PATH)
write_csv_atomic(full_spd_hyvarinen_diagnostics, FULL_SPD_HYVARINEN_DIAGNOSTIC_PATH)
write_csv_atomic(full_spd_ladder_raw, FULL_SPD_LADDER_RAW_PATH)
write_csv_atomic(full_spd_ladder_per_model, FULL_SPD_LADDER_PER_MODEL_PATH)
write_csv_atomic(full_spd_ladder_summary, FULL_SPD_LADDER_SUMMARY_PATH)
save_torch_atomic(
    {
        "config": full_spd_hyvarinen_config,
        "raw": full_spd_ladder_raw.to_dict("records"),
        "per_model": full_spd_ladder_per_model.to_dict("records"),
        "summary": full_spd_ladder_summary.to_dict("records"),
        "score_diagnostics": full_spd_hyvarinen_diagnostics.to_dict("records"),
    },
    FULL_SPD_LADDER_BUNDLE_PATH,
)
full_spd_ladder_summary


**Load the four-source results**


In [ ]:
required_full_spd_ladder_paths = [
    FULL_SPD_HYVARINEN_RAW_PATH,
    FULL_SPD_HYVARINEN_HISTORY_PATH,
    FULL_SPD_HYVARINEN_DIAGNOSTIC_PATH,
    FULL_SPD_LADDER_RAW_PATH,
    FULL_SPD_LADDER_PER_MODEL_PATH,
    FULL_SPD_LADDER_SUMMARY_PATH,
    FULL_SPD_LADDER_BUNDLE_PATH,
]
missing_full_spd_ladder_paths = [
    str(path) for path in required_full_spd_ladder_paths if not path.exists()
]
if missing_full_spd_ladder_paths:
    raise FileNotFoundError(
        f"Run the full-SPD Hyvarinen recovery cell first; missing {missing_full_spd_ladder_paths}"
    )

full_spd_hyvarinen_raw = pd.read_csv(FULL_SPD_HYVARINEN_RAW_PATH)
full_spd_hyvarinen_histories = pd.read_csv(FULL_SPD_HYVARINEN_HISTORY_PATH)
full_spd_hyvarinen_diagnostics = pd.read_csv(FULL_SPD_HYVARINEN_DIAGNOSTIC_PATH)
full_spd_ladder_raw = pd.read_csv(FULL_SPD_LADDER_RAW_PATH)
full_spd_ladder_per_model = pd.read_csv(FULL_SPD_LADDER_PER_MODEL_PATH)
full_spd_ladder_summary = pd.read_csv(FULL_SPD_LADDER_SUMMARY_PATH)


## Noise-family controls


**Family definitions and fitting rule**


In [ ]:
FULL_SPD_FAMILY_SPECS = [
    {
        "family": "full_generalized",
        "label": "Full generalized",
        "covariance_type": "full",
        "learn_beta": True,
        "beta_init": 1.6,
    },
    {
        "family": "forced_gaussian",
        "label": "Forced Gaussian, learned full SPD",
        "covariance_type": "full",
        "learn_beta": False,
        "beta_init": 2.0,
    },
    {
        "family": "forced_isotropic",
        "label": "Forced isotropic generalized",
        "covariance_type": "isotropic",
        "learn_beta": True,
        "beta_init": 1.6,
    },
    {
        "family": "forced_isotropic_gaussian",
        "label": "Forced isotropic Gaussian",
        "covariance_type": "isotropic",
        "learn_beta": False,
        "beta_init": 2.0,
    },
]
FULL_SPD_MISSPECIFIED_FAMILY_SPECS = FULL_SPD_FAMILY_SPECS[1:]
FULL_SPD_FAMILY_ORDER = [spec["family"] for spec in FULL_SPD_FAMILY_SPECS]
FULL_SPD_FAMILY_LABELS = {
    spec["family"]: spec["label"] for spec in FULL_SPD_FAMILY_SPECS
}
FULL_SPD_FAMILY_SPEC_BY_NAME = {spec["family"]: spec for spec in FULL_SPD_FAMILY_SPECS}
FULL_SPD_FAMILY_SOURCE_ORDER = [
    ("oracle_score", "importance_posterior", "Oracle score / importance posterior"),
    (
        "hyvarinen_score",
        "importance_posterior",
        "Hyvarinen score / importance posterior",
    ),
    ("oracle_score", "learned_posterior", "Oracle score / learned posterior"),
    ("hyvarinen_score", "learned_posterior", "Hyvarinen score / learned posterior"),
]

FULL_SPD_FAMILY_FIT_DIR = FULL_SPD_OUT_DIR / "family_fits"
FULL_SPD_FAMILY_FIT_DIR.mkdir(parents=True, exist_ok=True)
FULL_SPD_MISSPECIFIED_RAW_PATH = FULL_SPD_OUT_DIR / "full_spd_misspecified_fits_raw.csv"
FULL_SPD_MISSPECIFIED_HISTORY_PATH = (
    FULL_SPD_OUT_DIR / "full_spd_misspecified_histories.csv"
)
FULL_SPD_FAMILY_RAW_PATH = FULL_SPD_OUT_DIR / "full_spd_family_fits_raw.csv"
FULL_SPD_FAMILY_PER_MODEL_PATH = FULL_SPD_OUT_DIR / "full_spd_family_per_model.csv"
FULL_SPD_FAMILY_SUMMARY_PATH = FULL_SPD_OUT_DIR / "full_spd_family_summary.csv"
FULL_SPD_FAMILY_BUNDLE_PATH = FULL_SPD_OUT_DIR / "full_spd_family_results.pt"
FULL_SPD_FAMILY_TABLE_PATH = FULL_SPD_OUT_DIR / "full_spd_family_table.csv"
FULL_SPD_MAIN_TABLE_PATH = FULL_SPD_OUT_DIR / "full_spd_main_table.csv"


def fit_full_spd_family_from_frozen_sources(
    y,
    sigma,
    posterior_samples,
    score_estimator,
    *,
    family_spec,
    optimizer_seed,
):
    covariance_type = family_spec["covariance_type"]
    learn_beta = bool(family_spec["learn_beta"])
    beta_init = float(family_spec["beta_init"])

    if covariance_type == "isotropic" and not learn_beta:
        beta = torch.as_tensor(beta_init, device=y.device, dtype=y.dtype)
        Sigma = torch.eye(y.shape[1], device=y.device, dtype=y.dtype)
        field = scaled_energy_tweedie_field_from_samples(
            y,
            posterior_samples,
            beta=beta,
            Sigma=Sigma,
            sigma_values=sigma,
            sample_layout="bkd",
        )
        stats = profiled_score_reconstruction(
            score_estimator,
            field,
            lambda_bounds=(1e-4, 10.0),
        )
        return {
            "beta": float(beta),
            "lambda": float(stats["lambda"].detach().cpu()),
            "Sigma": Sigma.detach().cpu(),
            "objective": float(stats["objective"].detach().cpu()),
            "history": [],
        }

    result = fit_multiscale_profiled_noise_parameters(
        y,
        sigma,
        posterior_samples,
        score_estimator=score_estimator,
        covariance_type=covariance_type,
        beta_min=0.5,
        beta_init=beta_init,
        learn_beta=learn_beta,
        init_sigma=torch.eye(y.shape[1], device=y.device, dtype=y.dtype),
        lambda_bounds=(1e-4, 10.0),
        n_steps=FULL_SPD_N_STEPS,
        batch_size=FULL_SPD_BATCH_SIZE,
        lr=FULL_SPD_OPTIMIZER_LR,
        grad_clip=10.0,
        log_every=FULL_SPD_LOG_EVERY,
        seed=int(optimizer_seed),
        device=DEVICE,
        sample_layout="bkd",
    )
    return {
        "beta": float(result.beta),
        "lambda": float(result.lam),
        "Sigma": result.Sigma.detach().cpu(),
        "objective": float(result.objective),
        "history": result.history,
    }


**Fit the three restricted families**


Each restricted family reuses the frozen observations, score estimates, and posterior draws from the full generalized fit. The four families are full generalized, forced Gaussian with learned SPD geometry, forced isotropic generalized, and forced isotropic Gaussian.


In [ ]:
full_spd_misspecified_rows = []
full_spd_misspecified_history_frames = []

for replicate, (data_seed, score_spec) in enumerate(
    zip(FULL_SPD_DATA_SEEDS, FULL_SPD_HYVARINEN_SCORE_SPECS)
):
    score_seed = int(score_spec["training_seed"])
    score_model_record = full_spd_hyvarinen_model_cache[score_seed]
    data_path = FULL_SPD_DATA_DIR / f"paired_data_rep{replicate:02d}.pt"
    data_payload_cpu = torch.load(data_path, map_location="cpu", weights_only=False)
    data_payload = {
        key: value.to(device=DEVICE, dtype=DTYPE) if torch.is_tensor(value) else value
        for key, value in data_payload_cpu.items()
    }

    hyvarinen_score_estimation = predict_conditional_score(
        score_model_record["model"],
        data_payload["y_estimation"],
        data_payload["sigma_estimation"],
        batch_size=8192,
    ).detach()
    hyvarinen_score_evaluation = predict_conditional_score(
        score_model_record["model"],
        data_payload["y_evaluation"],
        data_payload["sigma_evaluation"],
        batch_size=8192,
    ).detach()
    score_sources = [
        {
            "score_source": "oracle_score",
            "score_training_seed": -1,
            "score_checkpoint": "independent_importance_score_reference",
            "estimation": data_payload["score_estimation"],
            "evaluation": data_payload["score_evaluation"],
        },
        {
            "score_source": "hyvarinen_score",
            "score_training_seed": score_seed,
            "score_checkpoint": str(score_model_record["checkpoint"]),
            "estimation": hyvarinen_score_estimation,
            "evaluation": hyvarinen_score_evaluation,
        },
    ]

    posterior_sources = [
        {
            "posterior_source": "importance_posterior",
            "model_name": "independent_importance_reference",
            "training_seed": -1,
            "checkpoint": "independent_importance_bank",
            "posterior_seed": int(data_seed) + 300,
            "estimation": data_payload["oracle_posterior_estimation"],
            "evaluation": data_payload["oracle_posterior_evaluation"],
        }
    ]
    for model_spec in FULL_SPD_MODEL_SPECS:
        posterior_training_seed = int(model_spec["training_seed"])
        oracle_stage_fit_path = (
            FULL_SPD_FIT_DIR
            / f"learned_seed{posterior_training_seed}_rep{replicate:02d}.pt"
        )
        oracle_stage_fit = torch.load(
            oracle_stage_fit_path, map_location="cpu", weights_only=False
        )
        if (
            "posterior_estimation" not in oracle_stage_fit
            or "posterior_evaluation" not in oracle_stage_fit
        ):
            raise RuntimeError(
                f"the oracle/learned-posterior stage fit does not contain saved posterior samples: {oracle_stage_fit_path}"
            )
        posterior_sources.append(
            {
                "posterior_source": "learned_posterior",
                "model_name": model_spec["model_name"],
                "training_seed": posterior_training_seed,
                "checkpoint": (
                    ROOT
                    / "models"
                    / "parameter_estimation"
                    / model_spec["model_name"]
                    / FULL_SPD_CHECKPOINT_NAME
                ),
                "posterior_seed": int(data_seed) + 10_000 + posterior_training_seed,
                "estimation": oracle_stage_fit["posterior_estimation"].to(
                    device=DEVICE, dtype=DTYPE
                ),
                "evaluation": oracle_stage_fit["posterior_evaluation"].to(
                    device=DEVICE, dtype=DTYPE
                ),
            }
        )

    oracle_score_power = (
        data_payload["score_evaluation"].square().sum(dim=1).mean().clamp_min(1e-12)
    )
    optimizer_seed = int(data_seed) + 400

    for score_source in score_sources:
        score_estimator_mse = (
            (score_source["evaluation"] - data_payload["score_evaluation"])
            .square()
            .sum(dim=1)
            .mean()
        )
        score_estimator_cosine = F.cosine_similarity(
            score_source["evaluation"],
            data_payload["score_evaluation"],
            dim=1,
            eps=1e-8,
        ).mean()

        for posterior_source in posterior_sources:
            posterior_tag = (
                "importance"
                if posterior_source["posterior_source"] == "importance_posterior"
                else f"learned_seed{posterior_source['training_seed']}"
            )
            score_tag = (
                "oracle"
                if score_source["score_source"] == "oracle_score"
                else f"hyvarinen_seed{score_seed}"
            )

            for family_spec in FULL_SPD_MISSPECIFIED_FAMILY_SPECS:
                family = family_spec["family"]
                fit_path = FULL_SPD_FAMILY_FIT_DIR / (
                    f"{family}_{score_tag}_{posterior_tag}_rep{replicate:02d}.pt"
                )
                if fit_path.exists():
                    fit_artifact = torch.load(
                        fit_path, map_location="cpu", weights_only=False
                    )
                    row = fit_artifact["row"]
                    history = pd.DataFrame(fit_artifact["history"])
                else:
                    fit = fit_full_spd_family_from_frozen_sources(
                        data_payload["y_estimation"],
                        data_payload["sigma_estimation"],
                        posterior_source["estimation"],
                        score_source["estimation"],
                        family_spec=family_spec,
                        optimizer_seed=optimizer_seed,
                    )
                    evaluation = evaluate_multiscale_noise_parameters(
                        data_payload["y_evaluation"],
                        data_payload["sigma_evaluation"],
                        posterior_source["evaluation"],
                        score_estimator=score_source["evaluation"],
                        reference_score=data_payload["score_evaluation"],
                        beta=fit["beta"],
                        lam=fit["lambda"],
                        Sigma=fit["Sigma"],
                        batch_size=FULL_SPD_BATCH_SIZE,
                        sample_layout="bkd",
                    )
                    fitted_sigma = fit["Sigma"].detach().cpu()
                    fitted_uv = uv_from_spd_2d(fitted_sigma)
                    fitted_eigenvalues = torch.linalg.eigvalsh(fitted_sigma)
                    row = {
                        "family": family,
                        "covariance_type": family_spec["covariance_type"],
                        "learn_beta": bool(family_spec["learn_beta"]),
                        "score_source": score_source["score_source"],
                        "posterior_source": posterior_source["posterior_source"],
                        "model_name": posterior_source["model_name"],
                        "training_seed": int(posterior_source["training_seed"]),
                        "score_training_seed": int(score_source["score_training_seed"]),
                        "replicate": int(replicate),
                        "data_seed": int(data_seed),
                        "estimation_observation_seed": int(
                            data_payload["estimation_observation_seed"]
                        ),
                        "evaluation_observation_seed": int(
                            data_payload["evaluation_observation_seed"]
                        ),
                        "posterior_seed": int(posterior_source["posterior_seed"]),
                        "optimizer_seed": optimizer_seed,
                        "checkpoint": str(posterior_source["checkpoint"]),
                        "score_checkpoint": score_source["score_checkpoint"],
                        "beta": float(fit["beta"]),
                        "lambda": float(fit["lambda"]),
                        "u": float(fitted_uv[0]),
                        "v": float(fitted_uv[1]),
                        "sigma_00": float(fitted_sigma[0, 0]),
                        "sigma_01": float(fitted_sigma[0, 1]),
                        "sigma_11": float(fitted_sigma[1, 1]),
                        "abs_beta_error": abs(float(fit["beta"]) - FULL_SPD_TRUE_BETA),
                        "abs_lambda_error": abs(
                            float(fit["lambda"]) - FULL_SPD_TRUE_LAMBDA
                        ),
                        "abs_u_error": abs(
                            float(fitted_uv[0]) - float(FULL_SPD_TRUE_UV[0])
                        ),
                        "abs_v_error": abs(
                            float(fitted_uv[1]) - float(FULL_SPD_TRUE_UV[1])
                        ),
                        "sigma_frobenius_error": float(
                            torch.linalg.matrix_norm(
                                fitted_sigma - FULL_SPD_TRUE_SIGMA.cpu()
                            )
                        ),
                        "fitted_condition_number": float(
                            fitted_eigenvalues.max() / fitted_eigenvalues.min()
                        ),
                        "estimation_objective": float(fit["objective"]),
                        "heldout_identity_residual": float(
                            evaluation["identity_residual"]
                        ),
                        "heldout_score_mse": float(evaluation["reference_mse"]),
                        "heldout_score_nmse": float(
                            evaluation["reference_mse"] / oracle_score_power
                        ),
                        "heldout_score_cosine": float(
                            F.cosine_similarity(
                                evaluation["prediction"],
                                data_payload["score_evaluation"],
                                dim=1,
                                eps=1e-8,
                            ).mean()
                        ),
                        "score_estimator_mse": float(score_estimator_mse),
                        "score_estimator_nmse": float(
                            score_estimator_mse / oracle_score_power
                        ),
                        "score_estimator_cosine": float(score_estimator_cosine),
                    }
                    history = pd.DataFrame(fit["history"])
                    history["family"] = family
                    history["score_source"] = score_source["score_source"]
                    history["posterior_source"] = posterior_source["posterior_source"]
                    history["model_name"] = posterior_source["model_name"]
                    history["training_seed"] = int(posterior_source["training_seed"])
                    history["score_training_seed"] = int(
                        score_source["score_training_seed"]
                    )
                    history["replicate"] = int(replicate)
                    save_torch_atomic(
                        {
                            "family_spec": family_spec,
                            "row": row,
                            "history": history.to_dict("records"),
                            "fit": fit,
                        },
                        fit_path,
                    )

                full_spd_misspecified_rows.append(row)
                if not history.empty:
                    full_spd_misspecified_history_frames.append(history)
                print(
                    f"family={family} replicate={replicate} "
                    f"score={score_source['score_source']} "
                    f"posterior={posterior_source['model_name']} "
                    f"beta={row['beta']:.4f} lambda={row['lambda']:.4f} "
                    f"u={row['u']:.4f} v={row['v']:.4f}"
                )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

full_spd_misspecified_raw = pd.DataFrame(full_spd_misspecified_rows)
full_spd_misspecified_histories = (
    pd.concat(full_spd_misspecified_history_frames, ignore_index=True)
    if full_spd_misspecified_history_frames
    else pd.DataFrame()
)
full_spd_full_generalized_raw = pd.read_csv(FULL_SPD_LADDER_RAW_PATH).assign(
    family="full_generalized"
)
full_spd_family_raw = pd.concat(
    [full_spd_full_generalized_raw, full_spd_misspecified_raw],
    ignore_index=True,
    sort=False,
)
family_order_map = {family: index for index, family in enumerate(FULL_SPD_FAMILY_ORDER)}
source_order_map = {
    (score_source, posterior_source): index
    for index, (score_source, posterior_source, _) in enumerate(
        FULL_SPD_FAMILY_SOURCE_ORDER
    )
}
full_spd_family_raw["_family_order"] = full_spd_family_raw["family"].map(
    family_order_map
)
full_spd_family_raw["_source_order"] = [
    source_order_map[(score_source, posterior_source)]
    for score_source, posterior_source in zip(
        full_spd_family_raw["score_source"],
        full_spd_family_raw["posterior_source"],
    )
]
full_spd_family_raw = (
    full_spd_family_raw.sort_values(
        ["_family_order", "_source_order", "replicate", "training_seed"]
    )
    .drop(columns=["_family_order", "_source_order"])
    .reset_index(drop=True)
)

full_spd_family_metrics = [
    "beta",
    "lambda",
    "u",
    "v",
    "abs_beta_error",
    "abs_lambda_error",
    "abs_u_error",
    "abs_v_error",
    "sigma_frobenius_error",
    "estimation_objective",
    "heldout_identity_residual",
    "heldout_score_mse",
    "heldout_score_nmse",
    "heldout_score_cosine",
    "score_estimator_mse",
    "score_estimator_nmse",
    "score_estimator_cosine",
]
full_spd_family_per_model_rows = []
full_spd_family_group_columns = [
    "family",
    "score_source",
    "posterior_source",
    "model_name",
    "training_seed",
]
for group_values, group in full_spd_family_raw.groupby(
    full_spd_family_group_columns, sort=False
):
    summary_row = dict(zip(full_spd_family_group_columns, group_values))
    summary_row["n_paired_replicates"] = int(group["replicate"].nunique())
    for metric in full_spd_family_metrics:
        values = group[metric].to_numpy(dtype=float)
        summary_row[f"{metric}_mean"] = float(values.mean())
        summary_row[f"{metric}_sd"] = (
            float(values.std(ddof=1)) if len(values) > 1 else 0.0
        )
    full_spd_family_per_model_rows.append(summary_row)
full_spd_family_per_model = pd.DataFrame(full_spd_family_per_model_rows)

full_spd_family_summary_rows = []
for family in FULL_SPD_FAMILY_ORDER:
    for score_source, posterior_source, source_label in FULL_SPD_FAMILY_SOURCE_ORDER:
        if posterior_source == "importance_posterior":
            units = full_spd_family_raw.loc[
                (full_spd_family_raw["family"] == family)
                & (full_spd_family_raw["score_source"] == score_source)
                & (full_spd_family_raw["posterior_source"] == posterior_source)
            ]
            metric_columns = {metric: metric for metric in full_spd_family_metrics}
            summary_unit = (
                "five separately generated data sets"
                if score_source == "oracle_score"
                else "five independently trained score models, each paired with a separately generated data set"
            )
        else:
            units = full_spd_family_raw.loc[
                (full_spd_family_raw["family"] == family)
                & (full_spd_family_raw["score_source"] == score_source)
                & (full_spd_family_raw["posterior_source"] == posterior_source)
            ]
            metric_columns = {metric: metric for metric in full_spd_family_metrics}
            summary_unit = (
                "fifteen fits crossing three independently trained posterior models"
                " with the five paired data runs"
            )
        if units.empty:
            raise RuntimeError(
                f"Missing full-SPD family rows for {family}, {score_source}, {posterior_source}"
            )

        summary_row = {
            "family": family,
            "family_label": FULL_SPD_FAMILY_LABELS[family],
            "score_source": score_source,
            "posterior_source": posterior_source,
            "source_label": source_label,
            "summary_unit": summary_unit,
            "n_units": int(len(units)),
        }
        for metric, column in metric_columns.items():
            values = units[column].to_numpy(dtype=float)
            summary_row[f"{metric}_mean"] = float(values.mean())
            summary_row[f"{metric}_sd"] = (
                float(values.std(ddof=1)) if len(values) > 1 else 0.0
            )
            summary_row[f"{metric}_sem"] = summary_row[f"{metric}_sd"] / math.sqrt(
                len(values)
            )
        full_spd_family_summary_rows.append(summary_row)
full_spd_family_summary = pd.DataFrame(full_spd_family_summary_rows)

write_csv_atomic(full_spd_misspecified_raw, FULL_SPD_MISSPECIFIED_RAW_PATH)
write_csv_atomic(full_spd_misspecified_histories, FULL_SPD_MISSPECIFIED_HISTORY_PATH)
write_csv_atomic(full_spd_family_raw, FULL_SPD_FAMILY_RAW_PATH)
write_csv_atomic(full_spd_family_per_model, FULL_SPD_FAMILY_PER_MODEL_PATH)
write_csv_atomic(full_spd_family_summary, FULL_SPD_FAMILY_SUMMARY_PATH)
save_torch_atomic(
    {
        "family_specs": FULL_SPD_FAMILY_SPECS,
        "raw": full_spd_family_raw.to_dict("records"),
        "per_model": full_spd_family_per_model.to_dict("records"),
        "summary": full_spd_family_summary.to_dict("records"),
    },
    FULL_SPD_FAMILY_BUNDLE_PATH,
)


In [ ]:
required_full_spd_family_paths = [
    FULL_SPD_MISSPECIFIED_RAW_PATH,
    FULL_SPD_MISSPECIFIED_HISTORY_PATH,
    FULL_SPD_FAMILY_RAW_PATH,
    FULL_SPD_FAMILY_PER_MODEL_PATH,
    FULL_SPD_FAMILY_SUMMARY_PATH,
    FULL_SPD_FAMILY_BUNDLE_PATH,
]
missing_full_spd_family_paths = [
    str(path) for path in required_full_spd_family_paths if not path.exists()
]
if missing_full_spd_family_paths:
    raise FileNotFoundError(
        f"Run the full-SPD misspecification cell first; missing {missing_full_spd_family_paths}"
    )

full_spd_misspecified_raw = pd.read_csv(FULL_SPD_MISSPECIFIED_RAW_PATH)
full_spd_misspecified_histories = pd.read_csv(FULL_SPD_MISSPECIFIED_HISTORY_PATH)
full_spd_family_raw = pd.read_csv(FULL_SPD_FAMILY_RAW_PATH)
full_spd_family_per_model = pd.read_csv(FULL_SPD_FAMILY_PER_MODEL_PATH)
full_spd_family_summary = pd.read_csv(FULL_SPD_FAMILY_SUMMARY_PATH)


**Main-paper table**


In [ ]:
full_spd_family_table_rows = []
for row in full_spd_family_summary.to_dict("records"):
    family_spec = FULL_SPD_FAMILY_SPEC_BY_NAME[row["family"]]
    beta_entry = (
        mean_sd_string(row["beta_mean"], row["beta_sd"], digits=4)
        if family_spec["learn_beta"]
        else "2.0000 (fixed)"
    )
    if family_spec["covariance_type"] == "isotropic":
        u_entry = "0.0000 (fixed)"
        v_entry = "0.0000 (fixed)"
    else:
        u_entry = mean_sd_string(row["u_mean"], row["u_sd"], digits=4)
        v_entry = mean_sd_string(row["v_mean"], row["v_sd"], digits=4)

    full_spd_family_table_rows.append(
        {
            "fitted family": row["family_label"],
            "source combination": row["source_label"],
            r"$\widehat{\beta}$": beta_entry,
            r"$\widehat{\lambda}$": mean_sd_string(
                row["lambda_mean"], row["lambda_sd"], digits=4
            ),
            r"$\widehat{u}$": u_entry,
            r"$\widehat{v}$": v_entry,
            r"$\|\widehat S-S^\star\|_F$": mean_sd_string(
                row["sigma_frobenius_error_mean"],
                row["sigma_frobenius_error_sd"],
                digits=4,
            ),
            "held-out score MSE": mean_sd_string(
                row["heldout_score_mse_mean"],
                row["heldout_score_mse_sd"],
                digits=4,
            ),
            "held-out score NMSE": mean_sd_string(
                row["heldout_score_nmse_mean"],
                row["heldout_score_nmse_sd"],
                digits=4,
            ),
        }
    )

full_spd_family_table = pd.DataFrame(full_spd_family_table_rows)
write_csv_atomic(full_spd_family_table, FULL_SPD_FAMILY_TABLE_PATH)

full_spd_main_rows = (full_spd_family_summary["family"] == "full_generalized") | (
    (full_spd_family_summary["score_source"] == "oracle_score")
    & (full_spd_family_summary["posterior_source"] == "importance_posterior")
)
full_spd_main_table = full_spd_family_table.loc[
    full_spd_main_rows.to_numpy(),
    [
        "fitted family",
        "source combination",
        r"$\widehat{\beta}$",
        r"$\widehat{\lambda}$",
        r"$\widehat{u}$",
        r"$\widehat{v}$",
        "held-out score NMSE",
    ],
].reset_index(drop=True)
write_csv_atomic(full_spd_main_table, FULL_SPD_MAIN_TABLE_PATH)


In [ ]:
full_spd_main_table = pd.read_csv(FULL_SPD_MAIN_TABLE_PATH)
full_spd_main_table


**Appendix table**


In [ ]:
full_spd_family_table = pd.read_csv(FULL_SPD_FAMILY_TABLE_PATH)
full_spd_family_table
